# Day 14 â€” Structured Streaming: Vehicle Battery Live â†’ Bronze
**Source:** Azure Event Hubs topic `vehicle-battery-live`  
**Sink:** `bronze/event-stream/vehicle_battery_live/` (Delta, append-only)  
**Checkpoint:** `bronze/_checkpoints/vehicle-battery-live/`  

Run cells 1â†’8 in order. Cell 8 blocks â€” the stream runs until you cancel the cell.

---
**Before running:** Attach this notebook to `dev-cluster` and make sure `send_vehicle_battery_events.py` is running on your local machine.

In [ ]:
# ── Cell 1: Read secrets from Key Vault ─────────────────────────────────────
# Reads the Event Hub LISTEN connection string and ADLS OAuth credentials.
# dbutils.secrets.get() returns [REDACTED] in logs — credentials are never exposed.
# Note: eventhub-vehicle-battery-listen-conn-str uses databricks-listen-policy (Listen only).
#       Do NOT use the producer key here — it has Send permission, not Listen.

EH_CONN_STR      = dbutils.secrets.get(scope="kv-ev-scope", key="eventhub-vehicle-battery-listen-conn-str")
SP_CLIENT_ID     = dbutils.secrets.get(scope="kv-ev-scope", key="sp-client-id")
SP_CLIENT_SECRET = dbutils.secrets.get(scope="kv-ev-scope", key="sp-client-secret")
SP_TENANT_ID     = dbutils.secrets.get(scope="kv-ev-scope", key="sp-tenant-id")
STORAGE_ACCOUNT  = dbutils.secrets.get(scope="kv-ev-scope", key="adls-account-name")

print("Secrets loaded successfully.")

In [ ]:
# â”€â”€ Cell 2: Configure ADLS Gen2 OAuth â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Sets Spark config so abfss:// paths on evdatalakedev authenticate via Service Principal OAuth.

spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_ID
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_SECRET
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token"
)

print(f"ADLS OAuth configured for: {STORAGE_ACCOUNT}")

In [ ]:
# ── Cell 3: Define paths and Event Hub config ────────────────────────────────

EVENTHUB_NAME   = "vehicle-battery-live"
CONSUMER_GROUP  = "$Default"

BRONZE_PATH     = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/event-stream/vehicle_battery_live/"
CHECKPOINT_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/vehicle-battery-live/"

EH_CONF = {
    "eventhubs.connectionString": sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(
        sc._javaSparkContext, EH_CONN_STR
    ),
    "eventhubs.consumerGroup": CONSUMER_GROUP,
    "eventhubs.maxEventsPerTrigger": 1000,
}

print(f"Bronze path:     {BRONZE_PATH}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Event Hub topic: {EVENTHUB_NAME}")
print(f"Consumer group:  {CONSUMER_GROUP}")

In [ ]:
# â”€â”€ Cell 4: Read stream from Event Hubs â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Each row from Event Hubs has: body (binary), enqueuedTime, offset, etc.
# The event payload is in the 'body' column as binary bytes.

raw_stream_df = (
    spark.readStream
    .format("eventhubs")
    .options(**EH_CONF)
    .load()
)

print("Stream schema (raw Event Hubs output):")
raw_stream_df.printSchema()

In [ ]:
# â”€â”€ Cell 5: Parse JSON body â†’ typed columns â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# 'body' is binary â€” decode to string, then parse JSON into individual columns.

from pyspark.sql.functions import (
    col, from_json, to_date, current_timestamp, lit
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, IntegerType
)

# Schema must match exactly what send_vehicle_battery_events.py sends
EVENT_SCHEMA = StructType([
    StructField("event_id",                    StringType(),  True),
    StructField("vehicle_id",                  StringType(),  True),
    StructField("session_id",                  StringType(),  True),
    StructField("station_id",                  StringType(),  True),
    StructField("charger_id",                  StringType(),  True),
    StructField("battery_pct",                 FloatType(),   True),
    StructField("charging_rate_kw",            FloatType(),   True),
    StructField("battery_temp_c",              FloatType(),   True),
    StructField("state_of_charge_target_pct",  IntegerType(), True),
    StructField("estimated_minutes_to_full",   IntegerType(), True),
    StructField("event_ts",                    StringType(),  True),
])

parsed_df = (
    raw_stream_df
    # decode binary body â†’ string
    .withColumn("body_str", col("body").cast("string"))
    # parse JSON string â†’ struct
    .withColumn("parsed", from_json(col("body_str"), EVENT_SCHEMA))
    # flatten struct to individual columns
    .select(
        col("parsed.event_id"),
        col("parsed.vehicle_id"),
        col("parsed.session_id"),
        col("parsed.station_id"),
        col("parsed.charger_id"),
        col("parsed.battery_pct"),
        col("parsed.charging_rate_kw"),
        col("parsed.battery_temp_c"),
        col("parsed.state_of_charge_target_pct"),
        col("parsed.estimated_minutes_to_full"),
        col("parsed.event_ts"),
        # Event Hub metadata â€” useful for debugging offset issues
        col("enqueuedTime").alias("eh_enqueued_time"),
        col("partition").alias("eh_partition"),
        col("offset").alias("eh_offset"),
    )
)

print("Parsed schema:")
parsed_df.printSchema()

In [ ]:
# â”€â”€ Cell 6: Add Bronze metadata columns â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Every Bronze table in this project gets these standard columns:
#   _ingestion_ts   â€” when the record landed in Bronze
#   _source         â€” which Event Hub topic it came from
#   _is_corrupt     â€” True if JSON parse returned all-null fields
# Also derive event_date (partition column) from event_ts.

from pyspark.sql.functions import to_date, when

bronze_df = (
    parsed_df
    .withColumn("_ingestion_ts", current_timestamp())
    .withColumn("_source",       lit(f"eventhub/{EVENTHUB_NAME}"))
    .withColumn(
        "_is_corrupt",
        # If event_id is null, JSON parse failed â€” mark as corrupt
        when(col("event_id").isNull(), lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "event_date",
        to_date(col("event_ts"))   # YYYY-MM-DD â€” used as partition column
    )
)

print("Bronze schema with metadata columns:")
bronze_df.printSchema()

In [ ]:
# â”€â”€ Cell 7: Write stream to Bronze Delta â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# - format: delta  (ACID, time travel, schema evolution)
# - outputMode: append  (Bronze is append-only â€” never modify raw data)
# - trigger: every 30 seconds  (micro-batch cadence)
# - partitionBy: event_date  (one folder per day â€” efficient for date-range queries)
# - checkpointLocation: saves Event Hub partition offsets per micro-batch
#   If the cluster restarts, streaming resumes from the last committed offset.

streaming_query = (
    bronze_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .partitionBy("event_date")
    .trigger(processingTime="30 seconds")
    .start(BRONZE_PATH)
)

print(f"Streaming query started. ID: {streaming_query.id}")
print(f"Status: {streaming_query.status}")
print()
print(f"Writing to: {BRONZE_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print()
print("Micro-batches run every 30 seconds.")
print("Start the Python producer now: python send_vehicle_battery_events.py")

In [ ]:
# â”€â”€ Cell 8: Keep stream alive + print progress every 30 seconds â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# This cell blocks until you cancel it (click Stop or interrupt kernel).
# Safe to cancel â€” checkpoint means the stream resumes exactly where it left off.

import time

print("Stream is live. Printing progress every 30 seconds.")
print("Cancel this cell to stop the stream.\n")

while streaming_query.isActive:
    progress = streaming_query.lastProgress
    if progress:
        num_input  = progress.get("numInputRows", 0)
        input_rate = progress.get("inputRowsPerSecond", 0.0)
        proc_rate  = progress.get("processedRowsPerSecond", 0.0)
        batch_id   = progress.get("batchId", "N/A")
        print(
            f"[Batch {batch_id}] "
            f"Rows this batch: {num_input:,} | "
            f"Input rate: {input_rate:.1f} rows/sec | "
            f"Processing rate: {proc_rate:.1f} rows/sec"
        )
    else:
        print("Waiting for first micro-batch...")
    time.sleep(30)

print("\nStreaming query stopped.")

In [ ]:
# â”€â”€ Cell 9 (OPTIONAL): Verify â€” read Bronze and check row count â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Run this in a SEPARATE notebook (not while Cell 8 is blocking) to verify data.
# Or cancel Cell 8 first, then run this cell.

verify_df = spark.read.format("delta").load(BRONZE_PATH)

print(f"Total rows in Bronze: {verify_df.count():,}")
print()

print("Latest 5 events:")
verify_df.orderBy(col("_ingestion_ts").desc()).show(5, truncate=False)

print("\nEvents per vehicle:")
verify_df.groupBy("vehicle_id").count().orderBy("vehicle_id").show()

print("\nBattery % range per vehicle:")
from pyspark.sql.functions import min as fmin, max as fmax, avg as favg
verify_df.groupBy("vehicle_id").agg(
    fmin("battery_pct").alias("min_battery_pct"),
    fmax("battery_pct").alias("max_battery_pct"),
    favg("battery_pct").alias("avg_battery_pct")
).orderBy("vehicle_id").show()

print("\nCorrupt events (_is_corrupt = True):")
verify_df.filter(col("_is_corrupt") == True).count()